```markdown
#### 1. Quantos pedidos únicos existem no dataset?
Como removemos duplicatas por `order_id` ao criar `df_limpo`, cada linha representa um pedido único.

```python
# Método recomendado
df_limpo["order_id"].nunique()

# Ou simplesmente:
len(df_limpo)
```

- `nunique()`: garante contagem real de IDs distintos.
- `len(df_limpo)`: funciona porque duplicatas já foram removidas.

---

#### 2. Status mais comum e menos comum
```python
status_counts = df_limpo["order_status"].value_counts()

status_mais_comum = status_counts.idxmax()
status_menos_comum = status_counts.idxmin()

status_mais_comum, status_menos_comum
```

- `value_counts()`: ordena do maior para o menor automaticamente.
- `idxmax()`: retorna o rótulo com maior frequência.

 **Esperado**:
- Mais comum → geralmente "delivered".
- Menos comum → normalmente "created" ou "approved".

---

#### 3. Criar `tempo_aprovacao_horas`
Tempo entre compra e aprovação:

```python
df_limpo["tempo_aprovacao_horas"] = (
    df_limpo["order_approved_at"] - df_limpo["order_purchase_timestamp"]
).dt.total_seconds() / 3600
```

- `.dt.total_seconds()`: transforma timedelta em segundos.
- Dividimos por 3600 para converter em horas.

---

#### 4. Pedidos em novembro de 2017
```python
filtro_nov_2017 = (
    (df_limpo["order_purchase_timestamp"].dt.year == 2017) &
    (df_limpo["order_purchase_timestamp"].dt.month == 11)
)

pedidos_nov_2017 = df_limpo[filtro_nov_2017]

len(pedidos_nov_2017)
```

- Usamos `.dt.year` e `.dt.month` porque já convertimos as datas para datetime.

 **Insight**: Novembro costuma ter pico por causa da Black Friday.

---

#### 5. Mês com maior volume de pedidos
```python
volume_mensal = (
    df_limpo
    .groupby(["ano_compra", "mes_compra"])
    .agg(total=("order_id", "count"))
    .reset_index()
)

volume_mensal.sort_values("total", ascending=False).head(1)
```

- O maior valor indicará o mês de maior volume.
 **Insight**: Normalmente ocorre no fim do ano (novembro/dezembro).

---

#### 6. Mediana do prazo — no prazo vs atrasado
```python
medianas = df_entregues.groupby("entregou_no_prazo")["prazo_entrega_dias"].median()

medianas
```

- `True`: pedidos no prazo.
- `False`: pedidos atrasados.

 **Esperado**: Pedidos atrasados terão mediana significativamente maior.

---

#### 7. Agrupar por mês e calcular total + prazo médio
```python
analise_mensal = (
    df_entregues
    .groupby("mes_compra")
    .agg(
        total_pedidos=("order_id", "count"),
        prazo_medio=("prazo_entrega_dias", "mean")
    )
    .sort_index()
)

analise_mensal
```

 **Padrão sazonal**:
- Fim do ano → maior volume.
- Meses com maior volume podem apresentar maior prazo médio (pressão logística).

---

#### 8. Pedidos com prazo acima de 60 dias
```python
prazo_60 = df_entregues[df_entregues["prazo_entrega_dias"] > 60]

len(prazo_60)

# Para investigar:
prazo_60[[
    "order_id",
    "prazo_entrega_dias",
    "atraso_dias"
]].sort_values("prazo_entrega_dias", ascending=False)
```

 **Possíveis causas**:
- Outliers extremos.
- Problemas logísticos.
- Erros de registro.
- Compras internacionais.

---

#### 9. Percentual por `faixa_prazo`
```python
df_entregues["faixa_prazo"].value_counts(normalize=True) * 100
```

- `normalize=True`: retorna proporção.
- Multiplicamos por 100 para percentual.

---

#### 10. Mês com pior taxa de pontualidade
```python
taxa_mensal = (
    df_entregues
    .groupby(["ano_compra", "mes_compra"])
    .agg(taxa=("entregou_no_prazo", "mean"))
    .reset_index()
)

taxa_mensal.sort_values("taxa").head(1)
```

- O menor valor indicará o mês com pior taxa de pontualidade.
```